# Evaluate on CPU

Loads every `preds_val_*.csv` and `preds_test_*.csv`, checks that the Image Index values and the label columns line up, and averages the available backbones. Thresholds are chosen per label on the validation split with Youden's J, then applied to the test split.

Test AUROC includes a 1000-draw patient-level bootstrap 95% interval. Sensitivity, specificity, accuracy, and F1 use the validation threshold. Sensitivity at 90% specificity is reported separately. Mass and Nodule are printed on their own.

ROC figures are saved at 200 dpi on a white background before `show()`, then checked so a blank file fails the notebook. Grad-CAM uses the single backbone with the best validation Mass AUROC, on 8 test images. That cell is the only one that needs a model; CPU is enough for 8 images.


In [ ]:
import os

N_BOOTSTRAP = 1000
SEED = 42
DATA_DIR = os.environ.get("DATA_DIR", "data/nih")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "data/output")

if os.environ.get("CXR_SEED"):
    SEED = int(os.environ["CXR_SEED"])
if os.environ.get("CXR_BOOTSTRAP"):
    N_BOOTSTRAP = int(os.environ["CXR_BOOTSTRAP"])
print("bootstrap", N_BOOTSTRAP, "seed", SEED)
print("DATA_DIR", DATA_DIR)
print("OUTPUT_DIR", OUTPUT_DIR)


In [ ]:
import sys
from pathlib import Path

def add_project_root():
    here = Path.cwd().resolve()
    candidates = [here, here.parent, *list(here.parents)[:4]]
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / "src" / "__init__.py").is_file() and (resolved / "src" / "paths.py").is_file():
            root = str(resolved)
            if root not in sys.path:
                sys.path.insert(0, root)
            print("Using code at", resolved)
            return resolved
    raise FileNotFoundError(
        "Could not find src/paths.py. Run this notebook from the project directory."
    )

add_project_root()


In [ ]:
from src.evaluate import run_evaluation

result = run_evaluation(
    n_bootstrap=N_BOOTSTRAP,
    seed=SEED,
    out_dir=OUTPUT_DIR,
    preprocessed_dir=OUTPUT_DIR,
)
result["focus"]
